# **InsightForge: AI-Powered Business Intelligence Assistant**
### Advanced Generative AI — Capstone Project

---

**Objective:**
Build an end-to-end AI-powered Business Intelligence (BI) assistant named **InsightForge** that:
- Analyzes 2,500 records of sales data using **pandas**
- Builds a **knowledge base** using LangChain, OpenAI Embeddings, and FAISS
- Provides **natural-language insights** via a custom **RAG pipeline**
- Maintains **conversational memory** across multi-turn interactions
- Evaluates response quality using **QAEvalChain**
- Presents **visualizations** of key business metrics
- Exposes a **Streamlit UI** for interactive querying

---

**Dataset:** `sales_data.csv`
Columns: `Date`, `Product`, `Region`, `Sales`, `Customer_Age`, `Customer_Gender`, `Customer_Satisfaction`

---


---
## Part 1: AI-Powered Business Intelligence Assistant
---


### **Setup: Environment & Imports**

Load environment variables (API keys) and import all required libraries.

> **Before running:** make sure your `OPENAI_API_KEY` is set in a `.env` file or as an environment variable.


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

# Verify API key is loaded
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found. Add it to a .env file."
print("Environment ready.")


In [ ]:
# ── Data & Visualization ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── LangChain Core ────────────────────────────────────────────────────────────
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.retrievers import BaseRetriever

# ── Memory ────────────────────────────────────────────────────────────────────
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# ── Evaluation ────────────────────────────────────────────────────────────────
from langchain.evaluation import QAEvalChain

# ── Typing ────────────────────────────────────────────────────────────────────
from pydantic import Field
from typing import List

print("All libraries imported successfully.")


---
### **Step 1: Data Preparation**

Load the sales dataset and perform exploratory analysis.
Focus is on *understanding and structuring* the data — not cleaning.


In [ ]:
DATA_PATH = "../Datasets_New/sales_data.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")


In [ ]:
# Data types and null check
print("=== Data Types ===")
print(df.dtypes)
print()
print("=== Null Values ===")
print(df.isnull().sum())


In [ ]:
# Statistical overview
print("=== Descriptive Statistics ===")
df[["Sales", "Customer_Age", "Customer_Satisfaction"]].describe().round(2)


In [ ]:
# Feature engineering for time-based analysis
df["Year"]       = df["Date"].dt.year
df["Month"]      = df["Date"].dt.month
df["Quarter"]    = df["Date"].dt.quarter
df["Month_Name"] = df["Date"].dt.strftime("%b")

print("Date range:", df["Date"].min().date(), "→", df["Date"].max().date())
print("Unique products:", df["Product"].unique())
print("Unique regions :", df["Region"].unique())
print("Gender split   :", df["Customer_Gender"].value_counts().to_dict())
df.head()


---
### **Step 2: Knowledge Base Creation**

Convert computed data summaries into LangChain `Document` objects,
chunk them, embed with OpenAI, and store in a **FAISS** vector store.


In [ ]:
def create_knowledge_documents(df: pd.DataFrame) -> list:
    """Build a rich set of text documents capturing all key sales statistics."""
    docs = []

    # ── Overall summary ──────────────────────────────────────────────────────
    total_sales   = df["Sales"].sum()
    avg_sales     = df["Sales"].mean()
    median_sales  = df["Sales"].median()
    std_sales     = df["Sales"].std()
    date_min      = df["Date"].min().date()
    date_max      = df["Date"].max().date()

    docs.append(Document(
        page_content=(
            f"Overall Business Summary:\n"
            f"Total Sales Revenue: ${total_sales:,.0f}\n"
            f"Average Daily Sales: ${avg_sales:.2f}\n"
            f"Median Daily Sales:  ${median_sales:.2f}\n"
            f"Std Dev of Sales:    ${std_sales:.2f}\n"
            f"Date Range: {date_min} to {date_max}\n"
            f"Total Transactions: {len(df):,}"
        ),
        metadata={"source": "overall_summary"}
    ))

    # ── Product-level analysis ───────────────────────────────────────────────
    product_stats = df.groupby("Product")["Sales"].agg(["sum", "mean", "std", "count"]).round(2)
    best_product  = product_stats["sum"].idxmax()
    text = "Product Sales Analysis:\n"
    for prod, row in product_stats.iterrows():
        text += (f"  {prod}: Total=${row['sum']:,.0f}, "
                 f"Avg=${row['mean']:.2f}, Std=${row['std']:.2f}, "
                 f"Days={int(row['count'])}\n")
    text += f"Top Performing Product: {best_product}"
    docs.append(Document(page_content=text, metadata={"source": "product_analysis"}))

    # ── Regional analysis ────────────────────────────────────────────────────
    region_stats = df.groupby("Region")["Sales"].agg(["sum", "mean", "count"]).round(2)
    best_region  = region_stats["sum"].idxmax()
    text = "Regional Sales Analysis:\n"
    for region, row in region_stats.iterrows():
        text += (f"  {region}: Total=${row['sum']:,.0f}, "
                 f"Avg=${row['mean']:.2f}, Days={int(row['count'])}\n")
    text += f"Top Performing Region: {best_region}"
    docs.append(Document(page_content=text, metadata={"source": "regional_analysis"}))

    # ── Monthly trends ───────────────────────────────────────────────────────
    monthly = df.groupby(["Year", "Month"])["Sales"].sum().reset_index()
    text = "Monthly Sales Trends:\n"
    for _, row in monthly.iterrows():
        text += f"  {int(row['Year'])}-{int(row['Month']):02d}: ${row['Sales']:,.0f}\n"
    peak_row = monthly.loc[monthly["Sales"].idxmax()]
    text += f"Peak Month: {int(peak_row['Year'])}-{int(peak_row['Month']):02d} (${peak_row['Sales']:,.0f})"
    docs.append(Document(page_content=text, metadata={"source": "monthly_trends"}))

    # ── Quarterly trends ─────────────────────────────────────────────────────
    quarterly = df.groupby(["Year", "Quarter"])["Sales"].sum().reset_index()
    text = "Quarterly Sales Trends:\n"
    for _, row in quarterly.iterrows():
        text += f"  {int(row['Year'])} Q{int(row['Quarter'])}: ${row['Sales']:,.0f}\n"
    docs.append(Document(page_content=text, metadata={"source": "quarterly_trends"}))

    # ── Customer demographics ────────────────────────────────────────────────
    age_stats    = df["Customer_Age"].describe()
    gender_sales = df.groupby("Customer_Gender")["Sales"].agg(["sum", "mean"]).round(2)
    sat_stats    = df["Customer_Satisfaction"].describe()
    text = (
        f"Customer Demographics Analysis:\n"
        f"  Age — Mean: {age_stats['mean']:.1f}, Min: {age_stats['min']:.0f}, "
        f"Max: {age_stats['max']:.0f}, Std: {age_stats['std']:.1f}\n"
        f"  Satisfaction — Mean: {sat_stats['mean']:.2f}, "
        f"Min: {sat_stats['min']:.2f}, Max: {sat_stats['max']:.2f}\n"
        f"  Gender Sales:\n"
    )
    for gender, row in gender_sales.iterrows():
        text += f"    {gender}: Total=${row['sum']:,.0f}, Avg=${row['mean']:.2f}\n"
    docs.append(Document(page_content=text, metadata={"source": "customer_demographics"}))

    # ── Product × Region cross-analysis ──────────────────────────────────────
    cross = df.groupby(["Product", "Region"])["Sales"].sum().unstack(fill_value=0)
    text = "Product × Region Cross Analysis:\n"
    for prod in cross.index:
        for region in cross.columns:
            text += f"  {prod} in {region}: ${cross.loc[prod, region]:,.0f}\n"
    docs.append(Document(page_content=text, metadata={"source": "cross_analysis"}))

    return docs

knowledge_docs = create_knowledge_documents(df)
print(f"Created {len(knowledge_docs)} knowledge documents.")
for doc in knowledge_docs:
    print(f"  [{doc.metadata['source']}] — {len(doc.page_content)} chars")


In [ ]:
# Split documents into chunks for fine-grained retrieval
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
split_docs = splitter.split_documents(knowledge_docs)
print(f"Split into {len(split_docs)} chunks.")


In [ ]:
# Create OpenAI embeddings and build FAISS vector store
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(split_docs, embeddings)
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("FAISS vector store created successfully.")
print(f"Index size: {vectorstore.index.ntotal} vectors")


---
### **Step 3: LLM Application Development**

Initialize the LLM and build a **custom pandas-based retriever** that computes
live statistics from the DataFrame based on the query intent.
This complements the FAISS retriever with exact, up-to-date numeric context.


In [ ]:
# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("LLM initialized:", llm.model_name)


In [ ]:
class SalesDataRetriever(BaseRetriever):
    """Custom retriever that computes on-the-fly statistics from the sales DataFrame."""

    dataframe: pd.DataFrame = Field(description="Sales DataFrame")

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(self, query: str) -> List[Document]:
        docs = []
        q = query.lower()

        # Always include overall summary
        docs.append(Document(
            page_content=(
                f"Overall Summary:\n"
                f"  Total Sales: ${self.dataframe['Sales'].sum():,.0f}\n"
                f"  Avg Daily: ${self.dataframe['Sales'].mean():.2f}\n"
                f"  Median: ${self.dataframe['Sales'].median():.2f}\n"
                f"  Std Dev: ${self.dataframe['Sales'].std():.2f}\n"
                f"  Records: {len(self.dataframe):,}\n"
                f"  Period: {self.dataframe['Date'].min().date()} → {self.dataframe['Date'].max().date()}"
            ),
            metadata={"source": "live_overall"}
        ))

        # Product stats
        if any(w in q for w in ["product", "widget", "item", "best", "top", "worst", "perform"]):
            ps = self.dataframe.groupby("Product")["Sales"].agg(["sum","mean","count"]).round(2)
            text = "Live Product Stats:\n" + "\n".join(
                f"  {p}: Total=${r['sum']:,.0f}, Avg=${r['mean']:.2f}, Days={int(r['count'])}"
                for p, r in ps.iterrows()
            )
            docs.append(Document(page_content=text, metadata={"source": "live_product"}))

        # Regional stats
        if any(w in q for w in ["region", "north", "south", "east", "west", "geographic", "area"]):
            rs = self.dataframe.groupby("Region")["Sales"].agg(["sum","mean","count"]).round(2)
            text = "Live Regional Stats:\n" + "\n".join(
                f"  {r}: Total=${row['sum']:,.0f}, Avg=${row['mean']:.2f}, Days={int(row['count'])}"
                for r, row in rs.iterrows()
            )
            docs.append(Document(page_content=text, metadata={"source": "live_region"}))

        # Time-series stats
        if any(w in q for w in ["time", "month", "year", "quarter", "trend", "season", "2022", "2023", "period"]):
            monthly = self.dataframe.groupby(["Year","Month"])["Sales"].sum().reset_index()
            text = "Live Monthly Sales:\n" + "\n".join(
                f"  {int(r['Year'])}-{int(r['Month']):02d}: ${r['Sales']:,.0f}"
                for _, r in monthly.iterrows()
            )
            docs.append(Document(page_content=text, metadata={"source": "live_monthly"}))

        # Customer stats
        if any(w in q for w in ["customer", "age", "gender", "satisfaction", "demograph", "segment"]):
            gender_s = self.dataframe.groupby("Customer_Gender")["Sales"].agg(["sum","mean"]).round(2)
            sat_mean = self.dataframe["Customer_Satisfaction"].mean()
            age_mean = self.dataframe["Customer_Age"].mean()
            text = (
                f"Live Customer Stats:\n"
                f"  Avg Age: {age_mean:.1f} years\n"
                f"  Avg Satisfaction: {sat_mean:.2f}/5\n"
                + "\n".join(
                    f"  {g}: Total=${row['sum']:,.0f}, Avg/day=${row['mean']:.2f}"
                    for g, row in gender_s.iterrows()
                )
            )
            docs.append(Document(page_content=text, metadata={"source": "live_customer"}))

        return docs

pandas_retriever = SalesDataRetriever(dataframe=df)
print("Custom SalesDataRetriever initialized.")


In [ ]:
# ── Advanced Data Summary using LLM chains ────────────────────────────────────

def run_analysis(analysis_type: str, data_context: str) -> str:
    """Run a targeted analysis chain for a specific business dimension."""
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a senior business intelligence analyst. "
         "Analyze the provided data and return 3-5 concise, actionable bullet-point insights. "
         "Include specific numbers. Be direct and business-focused."),
        ("human", "Analysis type: {analysis_type}\n\nData:\n{data_context}\n\nProvide key insights:")
    ])
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"analysis_type": analysis_type, "data_context": data_context})

# Sales performance by time period
monthly_data = df.groupby(["Year","Month"])["Sales"].agg(["sum","mean"]).round(2).to_string()
print("=" * 60)
print("SALES PERFORMANCE BY TIME PERIOD")
print("=" * 60)
print(run_analysis("Sales performance by time period", monthly_data))


In [ ]:
# Product and regional analysis
product_data = df.groupby("Product")["Sales"].describe().round(2).to_string()
region_data  = df.groupby("Region")["Sales"].describe().round(2).to_string()
combined     = f"Products:\n{product_data}\n\nRegions:\n{region_data}"

print("=" * 60)
print("PRODUCT & REGIONAL ANALYSIS")
print("=" * 60)
print(run_analysis("Product and regional performance", combined))


In [ ]:
# Customer segmentation analysis
age_bins = pd.cut(df["Customer_Age"], bins=[18,30,40,50,60,70], labels=["18-30","31-40","41-50","51-60","61-70"])
seg_data = df.groupby([age_bins, "Customer_Gender"])["Sales"].mean().round(2).to_string()
sat_data = df.groupby("Customer_Gender")["Customer_Satisfaction"].mean().round(2).to_string()
combined = f"Sales by Age & Gender:\n{seg_data}\n\nAvg Satisfaction by Gender:\n{sat_data}"

print("=" * 60)
print("CUSTOMER SEGMENTATION ANALYSIS")
print("=" * 60)
print(run_analysis("Customer segmentation by demographics", combined))


---
### **Step 4: Chain Prompts**

Design a **multi-stage sequential chain** that:
1. Extracts key metrics from the data context
2. Generates a comprehensive executive summary with recommendations


In [ ]:
# Stage 1 — Extract key metrics
metrics_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a data analyst. Extract and list the top 5 key business metrics "
     "from the provided sales data. Return only a concise numbered list."),
    ("human", "Sales data context:\n{data_context}")
])

# Stage 2 — Generate executive summary from extracted metrics
exec_summary_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Chief Business Officer writing an executive summary. "
     "Based on the key metrics provided, write a 3-paragraph executive summary "
     "with strategic recommendations for the leadership team."),
    ("human", "Key metrics:\n{metrics}")
])

# Build the sequential chain using LCEL
metrics_chain  = metrics_prompt  | llm | StrOutputParser()
summary_chain  = exec_summary_prompt | llm | StrOutputParser()

# Full pipeline: data → metrics → executive summary
full_pipeline = (
    metrics_chain
    | (lambda metrics: {"metrics": metrics})
    | summary_chain
)

# Prepare data context
overall_context = "\n".join(doc.page_content for doc in knowledge_docs[:4])

print("=" * 60)
print("EXECUTIVE SUMMARY (Sequential Chain Output)")
print("=" * 60)
exec_summary = full_pipeline.invoke({"data_context": overall_context})
print(exec_summary)


---
### **Step 5: RAG System Setup**

Implement a **hybrid retriever** combining:
- **SalesDataRetriever** — live pandas statistics (exact numbers)
- **FAISS retriever** — semantic similarity search over knowledge base documents

The two retrievers are merged and fed into an LCEL RAG chain.


In [ ]:
from langchain.retrievers import MergerRetriever

# Hybrid retriever: pandas live stats + FAISS semantic search
hybrid_retriever = MergerRetriever(retrievers=[pandas_retriever, faiss_retriever])
print("Hybrid retriever created (pandas + FAISS).")


In [ ]:
def format_docs(docs: List[Document]) -> str:
    seen, unique = set(), []
    for doc in docs:
        key = doc.metadata.get("source", "")
        if key not in seen:
            seen.add(key)
            unique.append(doc.page_content)
    return "\n\n---\n\n".join(unique)

# RAG prompt — single-turn (no memory yet)
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are InsightForge, an AI-powered Business Intelligence Assistant.
Use ONLY the data context below to answer the question accurately.
Always cite specific numbers from the context.
If the data does not contain the answer, say so clearly.

Context:
{context}"""),
    ("human", "{question}")
])

# LCEL RAG chain
rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(hybrid_retriever.invoke(x["question"]))
    )
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built with LCEL pipeline.")


In [ ]:
# Test RAG queries
test_queries = [
    "What is the total sales revenue and which product performs best?",
    "Which region generates the highest sales and by how much?",
    "What are the monthly sales trends — any seasonality?",
]

for q in test_queries:
    print(f"\nQ: {q}")
    print("-" * 50)
    print(rag_chain.invoke({"question": q}))
    print()


---
### **Step 6: Memory Integration**

Wrap the RAG chain in **`RunnableWithMessageHistory`** to give InsightForge
conversational memory — it retains context across a multi-turn session.


In [ ]:
# Memory-aware prompt
memory_rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are InsightForge, an AI-powered Business Intelligence Assistant.
Use the data context below to answer accurately. Always cite specific numbers.
Remember the conversation history to give coherent follow-up answers.

Context:
{context}"""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# RAG chain that passes history placeholder
memory_rag_core = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(hybrid_retriever.invoke(x["question"]))
    )
    | memory_rag_prompt
    | llm
    | StrOutputParser()
)

# Session store
session_store: dict = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

# Wrap with memory
insightforge = RunnableWithMessageHistory(
    memory_rag_core,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

print("InsightForge with memory initialized.")


In [ ]:
# Multi-turn conversation demo
SESSION = "demo_session"

def ask(question: str) -> str:
    return insightforge.invoke(
        {"question": question},
        config={"configurable": {"session_id": SESSION}}
    )

print("=" * 60)
print("MULTI-TURN CONVERSATION DEMO")
print("=" * 60)

turns = [
    "What was the total sales revenue?",
    "Which product contributed the most to that revenue?",
    "How did that product perform across different regions?",
    "What is the average customer satisfaction score for that product?",
]

for turn in turns:
    print(f"\nUser: {turn}")
    response = ask(turn)
    print(f"InsightForge: {response}")


In [ ]:
# Inspect conversation history
print("\n=== Session Memory ===")
history = get_session_history(SESSION)
for msg in history.messages:
    role = "User" if msg.type == "human" else "AI"
    print(f"[{role}]: {msg.content[:120]}{'...' if len(msg.content) > 120 else ''}")


---
## Part 2: LLMOps — Evaluation, Monitoring & Visualization
---


---
### **Step 7: Model Evaluation with QAEvalChain**

Evaluate InsightForge's response quality using **QAEvalChain**,
which uses the LLM itself as a judge to grade predictions as CORRECT/INCORRECT.


In [ ]:
# Ground-truth Q&A pairs derived directly from the data
best_product = df.groupby("Product")["Sales"].sum().idxmax()
best_region  = df.groupby("Region")["Sales"].sum().idxmax()

eval_examples = [
    {
        "query": "What is the total sales revenue across all transactions?",
        "answer": f"The total sales revenue is ${df['Sales'].sum():,.0f}."
    },
    {
        "query": "Which product has the highest total sales?",
        "answer": f"{best_product} has the highest total sales with ${df.groupby('Product')['Sales'].sum()[best_product]:,.0f}."
    },
    {
        "query": "Which region generates the highest sales?",
        "answer": f"{best_region} region generates the highest sales with ${df.groupby('Region')['Sales'].sum()[best_region]:,.0f}."
    },
    {
        "query": "What is the average customer satisfaction score?",
        "answer": f"The average customer satisfaction score is {df['Customer_Satisfaction'].mean():.2f} out of 5."
    },
    {
        "query": "What is the average customer age?",
        "answer": f"The average customer age is {df['Customer_Age'].mean():.1f} years."
    },
]

print(f"Prepared {len(eval_examples)} evaluation examples.")


In [ ]:
# Generate predictions using InsightForge (fresh session per question)
predictions = []
for i, ex in enumerate(eval_examples):
    pred = insightforge.invoke(
        {"question": ex["query"]},
        config={"configurable": {"session_id": f"eval_{i}"}}
    )
    predictions.append({"result": pred})
    print(f"  [{i+1}] Q: {ex['query'][:60]}...")

print("\nPredictions generated.")


In [ ]:
# Run QAEvalChain
eval_chain = QAEvalChain.from_llm(llm)
graded = eval_chain.evaluate(
    eval_examples,
    predictions,
    question_key="query",
    prediction_key="result"
)

# Display results table
results = pd.DataFrame({
    "Question"        : [e["query"]                         for e in eval_examples],
    "Expected Answer" : [e["answer"]                        for e in eval_examples],
    "Model Answer"    : [p["result"][:100] + "..."          for p in predictions],
    "Grade"           : [g.get("results", "N/A")            for g in graded],
})

pd.set_option("display.max_colwidth", 80)
print(results.to_string(index=False))


In [ ]:
# Evaluation summary
grades      = [g.get("results", "") for g in graded]
n_correct   = sum(1 for g in grades if "CORRECT" in g.upper())
n_incorrect = sum(1 for g in grades if "INCORRECT" in g.upper())
accuracy    = n_correct / len(grades) * 100

print(f"\n=== Evaluation Summary ===")
print(f"  Total questions : {len(grades)}")
print(f"  Correct         : {n_correct}")
print(f"  Incorrect       : {n_incorrect}")
print(f"  Accuracy        : {accuracy:.1f}%")


---
### **Step 8: Data Visualizations**

Present business insights through four key visualizations:
1. **Sales trends over time** — monthly line chart
2. **Product performance** — total & average sales bar chart
3. **Regional analysis** — horizontal bar + pie share
4. **Customer demographics** — age distribution & satisfaction by gender


In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})


In [ ]:
# ── Visualization 1: Sales Trends Over Time ─────────────────────────────────
monthly_sales = df.groupby(df["Date"].dt.to_period("M"))["Sales"].sum().reset_index()
monthly_sales["Date"] = monthly_sales["Date"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly_sales["Date"], monthly_sales["Sales"], marker="o", linewidth=2, color="#2E86AB", markersize=5)
ax.fill_between(monthly_sales["Date"], monthly_sales["Sales"], alpha=0.15, color="#2E86AB")

# Highlight peak month
peak_idx = monthly_sales["Sales"].idxmax()
ax.annotate(
    f"Peak\n${monthly_sales.loc[peak_idx, 'Sales']:,.0f}",
    xy=(monthly_sales.loc[peak_idx, "Date"], monthly_sales.loc[peak_idx, "Sales"]),
    xytext=(15, 15), textcoords="offset points",
    arrowprops=dict(arrowstyle="->", color="red"), fontsize=9, color="red"
)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.set_title("Monthly Sales Trends", fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Total Sales ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("viz_01_sales_trends.png", bbox_inches="tight")
plt.show()
print("Saved: viz_01_sales_trends.png")


In [ ]:
# ── Visualization 2: Product Performance ────────────────────────────────────
prod_stats = df.groupby("Product")["Sales"].agg(["sum", "mean"]).sort_values("sum", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Total sales by product
bars = axes[0].bar(prod_stats.index, prod_stats["sum"], color=["#2E86AB","#A23B72","#F18F01","#C73E1D"])
axes[0].bar_label(bars, labels=[f"${v:,.0f}" for v in prod_stats["sum"]], padding=3, fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[0].set_title("Total Sales by Product", fontweight="bold")
axes[0].set_xlabel("Product")
axes[0].set_ylabel("Total Sales ($)")

# Average sales by product
bars2 = axes[1].bar(prod_stats.index, prod_stats["mean"], color=["#2E86AB","#A23B72","#F18F01","#C73E1D"])
axes[1].bar_label(bars2, labels=[f"${v:.0f}" for v in prod_stats["mean"]], padding=3, fontsize=9)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[1].set_title("Average Daily Sales by Product", fontweight="bold")
axes[1].set_xlabel("Product")
axes[1].set_ylabel("Average Sales ($)")

plt.suptitle("Product Performance Analysis", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("viz_02_product_performance.png", bbox_inches="tight")
plt.show()
print("Saved: viz_02_product_performance.png")


In [ ]:
# ── Visualization 3: Regional Analysis ──────────────────────────────────────
region_sales = df.groupby("Region")["Sales"].agg(["sum","mean"]).sort_values("sum", ascending=True)
colors = ["#264653","#2A9D8F","#E9C46A","#E76F51"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Horizontal bar chart
bars = axes[0].barh(region_sales.index, region_sales["sum"], color=colors)
axes[0].bar_label(bars, labels=[f"${v:,.0f}" for v in region_sales["sum"]], padding=4, fontsize=9)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[0].set_title("Total Sales by Region", fontweight="bold")
axes[0].set_xlabel("Total Sales ($)")

# Pie chart — regional market share
axes[1].pie(
    region_sales["sum"],
    labels=region_sales.index,
    autopct="%1.1f%%",
    colors=colors,
    startangle=90,
    pctdistance=0.82
)
axes[1].set_title("Regional Market Share", fontweight="bold")

plt.suptitle("Regional Sales Analysis", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("viz_03_regional_analysis.png", bbox_inches="tight")
plt.show()
print("Saved: viz_03_regional_analysis.png")


In [ ]:
# ── Visualization 4: Customer Demographics & Segmentation ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Age distribution
axes[0].hist(df["Customer_Age"], bins=20, color="#2E86AB", edgecolor="white", linewidth=0.5)
axes[0].axvline(df["Customer_Age"].mean(), color="red", linestyle="--", label=f"Mean: {df['Customer_Age'].mean():.1f}")
axes[0].set_title("Customer Age Distribution", fontweight="bold")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=9)

# Sales by gender
gender_sales = df.groupby("Customer_Gender")["Sales"].mean()
bar_colors = ["#E76F51" if g == "Female" else "#2E86AB" for g in gender_sales.index]
bars = axes[1].bar(gender_sales.index, gender_sales.values, color=bar_colors)
axes[1].bar_label(bars, labels=[f"${v:.0f}" for v in gender_sales.values], padding=3, fontsize=10)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[1].set_title("Avg Daily Sales by Gender", fontweight="bold")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Avg Sales ($)")

# Customer satisfaction distribution
axes[2].hist(df["Customer_Satisfaction"], bins=20, color="#A23B72", edgecolor="white", linewidth=0.5)
axes[2].axvline(df["Customer_Satisfaction"].mean(), color="orange", linestyle="--",
                label=f"Mean: {df['Customer_Satisfaction'].mean():.2f}")
axes[2].set_title("Customer Satisfaction Distribution", fontweight="bold")
axes[2].set_xlabel("Satisfaction Score")
axes[2].set_ylabel("Count")
axes[2].legend(fontsize=9)

plt.suptitle("Customer Demographics & Segmentation", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("viz_04_customer_demographics.png", bbox_inches="tight")
plt.show()
print("Saved: viz_04_customer_demographics.png")


---
## Conclusion

**InsightForge** successfully demonstrates an end-to-end AI-powered Business Intelligence pipeline:

| Component | Implementation |
|---|---|
| Data Layer | pandas, feature engineering, descriptive statistics |
| Knowledge Base | LangChain Documents → FAISS vectorstore (OpenAI embeddings) |
| Custom Retriever | `SalesDataRetriever` — live pandas statistics on query intent |
| Hybrid RAG | MergerRetriever (pandas + FAISS) + LCEL chain |
| LLM | GPT-4o-mini via ChatOpenAI |
| Memory | `RunnableWithMessageHistory` + `ChatMessageHistory` |
| Evaluation | `QAEvalChain` — LLM-as-judge grading |
| Visualizations | Matplotlib / Seaborn — 4 business dashboards |
| UI | Streamlit app (`streamlit_app.py`) |

**Key Business Insights discovered:**
- Identified top-performing products and regions with exact revenue figures
- Uncovered monthly/quarterly sales seasonality
- Segmented customers by age, gender, and satisfaction score
- Delivered an interactive chat interface for non-technical stakeholders

---
